In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
os.chdir("..")

In [3]:
import torch
import numpy as np
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN, THINK_START_TOKEN



os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.bfloat16
device   = 'cuda'
model_id = "Qwen/QwQ-32B"

In [4]:
from pathlib import Path

cur_dir = Path(".").absolute()


def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/qwq-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

In [5]:
task_name = "plan_generation_po"
domain_name = f"blocksworld_mystery"
eval_results = load_dataset_from_file(domain_name, task_name)["instances"]
eval_results = {x["dataset_idx"]: x for x in eval_results}

In [6]:
tokenizer = initialize_tokenizer(model_id)

In [7]:
clean_type = "4-blocks-small"
mystery_type = "mystery-24k"

dataset_clean = load_dataset(f"dmitriihook/qwq-32b-planning-{clean_type}")["train"]
dataset_mystery = load_dataset(f"dmitriihook/qwq-32b-planning-{mystery_type}")["train"]

In [8]:
phrases = [
    "pick up",
    "put down",
    "stack",
    "unstack"
]

In [9]:
phrases_m = [
    "attack",
    "succumb",
    "overcome",
    "feast"
]

In [11]:
def extract_phrase_end_positions(tokens: torch.Tensor, phrase: str) -> list[str]:
    """Find end of the phrase token positions"""
    tokens = tokens.squeeze()

    phrase_tokens = tokenizer.encode(" " + phrase)
    phrase_tokens_cap = tokenizer.encode(" " + phrase.capitalize())
    phrase_tokens_nl = tokenizer.encode("\n" + phrase.capitalize())
    phrase_tokens_nll = tokenizer.encode("\n\n" + phrase.capitalize())


    positions = []

    for phts in [phrase_tokens, phrase_tokens_cap, phrase_tokens_nl, phrase_tokens_nll]:
        presence_mask = torch.ones_like(tokens)
        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]

        positions += (torch.where(presence_mask)[0] + len(phts) - 1).tolist()
    
    return sorted(positions)


def extract_phrase_all_positions(tokens: torch.Tensor, phrase: str) -> list[str]:
    """Find all token positions of the phrase"""
    tokens = tokens.squeeze()

    phrase_tokens = tokenizer.encode(" " + phrase)
    phrase_tokens_cap = tokenizer.encode(" " + phrase.capitalize())
    phrase_tokens_nl = tokenizer.encode("\n" + phrase.capitalize())
    phrase_tokens_nl1 = tokenizer.encode("\n" + phrase)
    phrase_tokens_nl2 = tokenizer.encode("\n\n" + phrase.capitalize())
    phrase_tokens_nl3 = tokenizer.encode("\n\n" + phrase)

    positions = []

    for phts in [phrase_tokens, phrase_tokens_cap, phrase_tokens_nl, phrase_tokens_nl1, phrase_tokens_nl2, phrase_tokens_nl3]:
        presence_mask = torch.ones_like(tokens)
        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]


        for i in range(len(phts)):
            positions += (torch.where(presence_mask)[0] + i).tolist()
    
    return sorted(positions)

def extract_phrase_before_positions(tokens: torch.Tensor, phrase: str) -> list[str]:
    """Find all token positions before the phrase"""
    tokens = tokens.squeeze()

    phrase_tokens = [
        tokenizer.encode(" " + phrase),
        tokenizer.encode(" " + phrase.capitalize()),
        tokenizer.encode("\n" + phrase)[1:],
        tokenizer.encode("\n" + phrase.capitalize())[1:],
        tokenizer.encode("\n\n" + phrase)[1:],
        tokenizer.encode("\n\n" + phrase.capitalize())[1:],
    ]

    positions = []

    for phts in phrase_tokens:
        presence_mask = torch.ones_like(tokens)
        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]

        positions += (torch.where(presence_mask)[0] - 1).tolist()

    return sorted(list(set(positions)))

In [12]:
model     = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=compute_dtype, attn_implementation="sdpa", 
                                                device_map="auto")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [13]:
from collections import defaultdict

def collect_hidden_states(ids: list[int], dataset, layers: list[int]) -> dict[int, dict]:
    hidden_states = defaultdict(dict)
    for idx in tqdm(ids):
        row = dataset[idx]
        tokens = tokenize_blocksworld_generation(tokenizer, row)
        with torch.no_grad():
            hs = model(tokens.to(device), output_hidden_states=True).hidden_states
            for layer in layers:
                hidden_states[idx][layer] = hs[layer][0].cpu().to(torch.float16).numpy()

    return hidden_states

In [14]:
n_rows = 40
layer = 47

In [15]:
clean_ids = list(range(n_rows))
mystery_ids = []
for idx in range(303):
    if eval_results[idx]["llm_correct"]:
        mystery_ids.append(idx)
    if len(mystery_ids) == n_rows:
        break

len(mystery_ids)

40

In [16]:
hidden_states_clean = collect_hidden_states(clean_ids, dataset_clean, [layer])

  0%|          | 0/40 [00:00<?, ?it/s]

In [17]:
hidden_states_mystery = collect_hidden_states(mystery_ids, dataset_mystery, [layer])

  0%|          | 0/40 [00:00<?, ?it/s]

In [18]:
def collect_mean_representations(ids: list[int], dataset, hidden_states, layer: int, phrases: list[str], n_pos: int=3) -> np.ndarray:
    reprs = defaultdict(list)
    for idx in tqdm(ids):
        row = dataset[idx]
        hs: torch.Tensor = hidden_states[idx][layer]

        generation = row["generation"]
        text = generation.split("</think>")[0]

        tokens = tokenize_blocksworld_generation(tokenizer, row, text)[0]

        for phrase in phrases:
            positions = extract_phrase_before_positions(tokens, phrase)

            prs = hs[positions[-n_pos - 1:]]
            reprs[phrase].append(prs.mean(0))

    return reprs

In [19]:
clean_reprs = collect_mean_representations(clean_ids, dataset_clean, hidden_states_clean, layer, phrases, n_pos=10)
mystery_reprs = collect_mean_representations(mystery_ids, dataset_mystery, hidden_states_mystery, layer, phrases_m, n_pos=10)

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

In [20]:
mean_reprs_clean = {k: np.stack(v).mean(0) for k, v in clean_reprs.items()}
mean_reprs_mystery = {k: np.stack(v).mean(0) for k, v in mystery_reprs.items()}

mean_clean = np.stack([mean_reprs_clean[p] for p in phrases], axis=0).mean(0)
mean_mystery = np.stack([mean_reprs_mystery[p] for p in phrases_m], axis=0).mean(0)

In [26]:

idx = 0
row = dataset_mystery[idx]

tokens = tokenize_blocksworld_generation(tokenizer, row, "")[:, :-2]

action_postions = {
   phrase: extract_phrase_before_positions(tokens, phrase)
    for phrase in phrases_m
}

In [27]:
{
   phrase: extract_phrase_all_positions(tokens[0, :100], phrase)
    for phrase in phrases_m
}

{'attack': [21, 56, 77, 93],
 'succumb': [32, 33],
 'overcome': [37, 38],
 'feast': [25]}

In [28]:
phrase = "attack"

print(
    tokenizer.encode(" " + phrase),
    tokenizer.encode(" " + phrase.capitalize()),
    tokenizer.encode("\n" + phrase.capitalize()),
    tokenizer.encode("\n" + phrase),
    tokenizer.encode("\n\n" + phrase.capitalize())
)

[3359] [20790] [198, 28602] [198, 20566] [271, 28602]


In [29]:
_tokens = tokens[0].clone()

for ip, ph in enumerate(phrases_m):
    positions = action_postions[ph]
    r = mean_reprs_mystery[phrases_m[ip]]
    _tokens[positions] = 1234

# tokens[0, 25:50]

In [30]:
print(tokenizer.decode(_tokens))

<|im_start|>user
I am playing with a set of objects. Here are the actions I can do

Item Attack object
Item Feast object from another object
Item Succumb object
Item Overcome object from another object

I have the following restrictions on my actions:
    ToItem Attack action, the following facts need to be true: Province object, Planet object, Harmony.
   Item Attack action is performed the following facts will be true: Pain object.
   Item Attack action is performed the following facts will be false: Province object, Planet object, Harmony.
    ToItem Succumb action, the following facts need to be true: Pain object.
   Item Succumb action is performed the following facts will be true: Province object, Planet object, Harmony.    
   Item Succumb action is performed the following facts will be false: Pain object.
    ToItem Overcome action, the following needs to be true: Province other object, Pain object.
   Item Overcome action is performed the following will be true: Harmony, Provi

In [31]:
from collections import OrderedDict


def forward_hook(module, input, output):
    """Replace output with the mean representation"""
    output = output[0]
    if output.shape[1] == 1:
        return (output,)
    
    output = output[0]    

    print("asdasd")
    
    for ip, ph in enumerate(phrases_m):
        positions = action_postions[ph]
        r = mean_reprs_mystery[phrases_m[ip]]
        output[positions] = torch.tensor(r, dtype=output.dtype, device=output.device)

    return (output.unsqueeze(0),)
    

for m in model.modules():
    m._forward_hooks = OrderedDict()
    
model.model.layers[0].register_forward_hook(forward_hook)

with torch.no_grad():
    enc = model.generate(tokens.to(device), do_sample=False, max_new_tokens=10000, temperature=None, top_p=None, top_k=None, use_cache=True)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


asdasd


In [32]:
print(tokenizer.decode(enc[0]))

<|im_start|>user
I am playing with a set of objects. Here are the actions I can do

   Attack object
   Feast object from another object
   Succumb object
   Overcome object from another object

I have the following restrictions on my actions:
    To perform Attack action, the following facts need to be true: Province object, Planet object, Harmony.
    Once Attack action is performed the following facts will be true: Pain object.
    Once Attack action is performed the following facts will be false: Province object, Planet object, Harmony.
    To perform Succumb action, the following facts need to be true: Pain object.
    Once Succumb action is performed the following facts will be true: Province object, Planet object, Harmony.    
    Once Succumb action is performed the following facts will be false: Pain object.
    To perform Overcome action, the following needs to be true: Province other object, Pain object.
    Once Overcome action is performed the following will be true: Harmo

In [167]:
len(tokenize_blocksworld_generation(tokenizer, row)[0])

11221

In [176]:
len(tokenizer.encode(row["generation"]))

10549

In [177]:
print(row["generation"])

Okay, let's see. I need to solve this problem where the initial conditions are given, and I have to come up with a plan using the actions provided to reach the goal. Let me start by understanding the problem step by step.

First, let me restate the initial conditions and the goal to make sure I have them right. The initial conditions are:

- Block C craves Block B (so "Object Craves other object" for C and B)
- Harmony exists (Harmony is true)
- Planet Block A, Planet Block B, Planet Block D (so all these blocks are on a planet)
- Province Block A, Province Block C, Province Block D (so these are in a province, but what about Block B? Wait, the initial conditions don't mention province for Block B. Wait, the problem says "province Block A, province Block C and province Block D." So Block B's province status isn't mentioned here. Hmm, maybe it's not in a province? Or maybe it's a typo? Wait, the problem says "province Block A, province Block C and province Block D." So Block B is not in